# Decision Trees

A complete guide: intuition, mathematics, and implementation from scratch.

## Table of Contents

1. [Intuition](#intuition)
2. [Tree Structure](#tree-structure)
3. [Setup & Imports](#imports)
4. [The Core Problem: How to Choose a Split?](#core-problem)
5. [Gini Impurity](#gini)
6. [Entropy and Information Gain](#entropy)
7. [Variance Reduction (Regression)](#variance)
8. [Computing the Split Gain](#split-gain)
9. [Finding the Best Split](#best-split)
10. [Building the Tree Recursively](#recursive-build)
11. [Complete Decision Tree Implementation](#implementation)
12. [Prediction: Traversing the Tree](#prediction)
13. [Demo: Classification (Iris)](#classification-demo)
14. [Visualizing the Decision Boundary](#decision-boundary)
15. [Overfitting vs. Regularization](#overfitting)
16. [Demo: Regression](#regression-demo)
17. [Feature Importance](#feature-importance)
18. [Complexity Analysis](#complexity)
19. [Advantages and Disadvantages](#pros-cons)
20. [Summary](#summary)
21. [Sources](#sources)

<a id="intuition"></a>
## 1. Intuition

A **decision tree** is one of the most intuitive machine learning algorithms. It mimics how humans make decisions: by asking a sequence of yes/no questions.

**Real-world analogy — diagnosing a patient:**

```
Does the patient have a fever?
├── YES → Does the patient have a cough?
│         ├── YES → Likely flu
│         └── NO  → Likely bacterial infection
└── NO  → Does the patient have a rash?
          ├── YES → Likely allergy
          └── NO  → Likely healthy
```

Each **internal node** tests a feature. Each **branch** represents the outcome of that test. Each **leaf node** holds a prediction.

The machine learning problem is: **given a dataset, how do we automatically learn the best sequence of questions?**

That is the core challenge. The answer lies in defining what "best" means — i.e., choosing a **splitting criterion**.

<a id="tree-structure"></a>
## 2. Tree Structure

A decision tree is a recursive data structure. Each node is either:

- **Internal node**: stores a splitting rule `(feature_index, threshold)` and two children
- **Leaf node**: stores the final prediction (class label or mean value)

Key terminology:

| Term | Meaning |
|---|---|
| **Root** | Top node; represents the full dataset |
| **Depth** | Number of levels from root to a node |
| **Pure node** | All samples belong to the same class |
| **Impurity** | Measure of how mixed the classes are in a node |

The algorithm builds the tree **top-down** and **greedily**: at each node, it picks the split that maximally reduces impurity in the resulting children.

<a id="imports"></a>
## 3. Setup & Imports

We now load the required libraries. All implementations use only NumPy, `collections.Counter`, and scikit-learn (for comparison only).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
from sklearn.datasets import load_iris, make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

np.random.seed(42)
print("Imports OK")

<a id="core-problem"></a>
## 4. The Core Problem: How to Choose a Split?

At each node we have a dataset $D$ with $n$ samples. We want to split it into two subsets $D_L$ and $D_R$ by testing a feature $x_j$ against a threshold $t$:

$$D_L = \{(x, y) \in D \mid x_j \leq t\}, \quad D_R = \{(x, y) \in D \mid x_j > t\}$$

We want the split that **maximally reduces impurity**. The **information gain** (or impurity reduction) of a split is:

$$\text{Gain}(D, j, t) = I(D) - \frac{|D_L|}{|D|} I(D_L) - \frac{|D_R|}{|D|} I(D_R)$$

where $I(\cdot)$ is an **impurity measure**. We pick the $(j, t)$ pair that maximizes the gain.

Three classic choices for $I$:
1. **Gini Impurity** — used by CART (sklearn default for classification)
2. **Entropy / Information Gain** — used by ID3, C4.5
3. **Variance Reduction** — used for regression trees

<a id="gini"></a>
## 5. Splitting Criterion 1: Gini Impurity

### Intuition

Gini impurity measures the probability that a randomly chosen sample would be **misclassified** if we labeled it according to the class distribution of the node.

Imagine drawing a sample at random from node $D$, then assigning it a random class proportional to the class frequencies. What is the probability of getting the wrong class?

- If all samples are the same class → we always pick the right class → impurity = **0** (pure node)
- If samples are split 50/50 → we're wrong half the time → impurity = **0.5** (maximum for binary)

### Formula

Let $p_k$ be the proportion of samples in class $k$ at node $D$:

$$\text{Gini}(D) = 1 - \sum_{k=1}^{K} p_k^2$$

**Derivation intuition:** The probability of drawing class $k$ is $p_k$. The probability it is *not* class $k$ is $(1 - p_k)$. The expected misclassification rate is $\sum_k p_k (1 - p_k) = 1 - \sum_k p_k^2$.

**Range:** $[0, 1 - 1/K]$. For binary classification: $[0, 0.5]$.

**Example:** Node with classes [A, A, A, B, B] → $p_A = 3/5$, $p_B = 2/5$

$$\text{Gini} = 1 - (3/5)^2 - (2/5)^2 = 1 - 0.36 - 0.16 = 0.48$$

We define `gini_impurity` as a function of a label array. It counts class frequencies and applies the formula $1 - \sum_k p_k^2$.

In [ ]:
def gini_impurity(y):
    """
    Gini impurity of an array of labels y.
    Returns 0 for a pure node, up to (1 - 1/K) for uniform distribution.
    """
    n = len(y)
    if n == 0:
        return 0.0
    counts = Counter(y)
    return 1.0 - sum((count / n) ** 2 for count in counts.values())

We verify the function against three manually computed examples: a pure node (Gini = 0), a mixed binary node (Gini ≈ 0.48), and a uniform three-class node (Gini = 2/3).

In [ ]:
# Manual examples
y_pure     = [1, 1, 1, 1, 1]       # all same class → Gini = 0
y_mixed    = [1, 1, 1, 2, 2]       # 3/5 and 2/5 → Gini ≈ 0.48
y_uniform  = [1, 1, 2, 2, 3, 3]   # equal split 3 classes → Gini = 2/3

print(f"Pure node:     Gini = {gini_impurity(y_pure):.4f}  (expected 0.0000)")
print(f"Mixed 3/5:     Gini = {gini_impurity(y_mixed):.4f}  (expected 0.4800)")
print(f"Uniform 3-cls: Gini = {gini_impurity(y_uniform):.4f}  (expected 0.6667)")

We plot Gini impurity as a function of the proportion $p$ of the positive class in the binary case. The parabola peaks at $p = 0.5$ (maximum uncertainty) and is zero at the pure extremes.

In [ ]:
# Visualize Gini as a function of p (binary case)
p = np.linspace(0, 1, 200)
gini = 2 * p * (1 - p)   # equivalent to 1 - p^2 - (1-p)^2 for binary

plt.figure(figsize=(7, 4))
plt.plot(p, gini, 'b-', linewidth=2, label='Gini = 2p(1-p)')
plt.axvline(0.5, color='gray', linestyle='--', alpha=0.6, label='p=0.5 (max impurity)')
plt.xlabel('p (proportion of class 1)')
plt.ylabel('Gini Impurity')
plt.title('Gini Impurity — Binary Classification')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<a id="entropy"></a>
## 6. Splitting Criterion 2: Entropy and Information Gain

### Intuition

Entropy comes from **information theory** (Shannon, 1948). It measures **surprise** or **uncertainty** in a distribution.

- If one class has probability ~1 → we always know what's coming → low entropy (low surprise)
- If all classes are equally likely → we have no idea → high entropy (maximum surprise)

Intuitively: *how many bits do we need to encode the class label of a random sample from this node?*

### Formula

$$H(D) = -\sum_{k=1}^{K} p_k \log_2 p_k$$

Convention: $0 \log 0 = 0$ (by L'Hôpital's rule).

**Range:** $[0, \log_2 K]$. For binary: $[0, 1]$ bit.

### Information Gain

Information Gain is how much entropy we *remove* by splitting:

$$\text{IG}(D, j, t) = H(D) - \frac{|D_L|}{|D|} H(D_L) - \frac{|D_R|}{|D|} H(D_R)$$

### Gini vs. Entropy

Both measure impurity and tend to give very similar trees. Key differences:

| | Gini | Entropy |
|---|---|---|
| Range (binary) | [0, 0.5] | [0, 1] |
| Computation | Faster (no log) | Slightly slower |
| Bias | Slightly favors larger partitions | Slightly favors balanced splits |
| sklearn default | Yes (classification) | No (but available) |

In practice: **the choice rarely matters** for final accuracy.

We implement Shannon entropy using base-2 logarithms. The function accumulates $-p \log_2 p$ for each class, with the convention that empty nodes return zero.

In [ ]:
def entropy(y):
    """
    Shannon entropy of label array y (in bits, base-2 log).
    Returns 0 for pure node, log2(K) for uniform K-class distribution.
    """
    n = len(y)
    if n == 0:
        return 0.0
    counts = Counter(y)
    result = 0.0
    for count in counts.values():
        p = count / n
        result -= p * np.log2(p)
    return result

We verify entropy on the same three example arrays used for Gini, checking that a pure node gives 0 bits, the mixed node gives ≈ 0.971 bits, and the uniform 3-class node gives $\log_2 3$ bits.

In [ ]:
print(f"Pure node:     H = {entropy(y_pure):.4f}  (expected 0.0000)")
print(f"Mixed 3/5:     H = {entropy(y_mixed):.4f}  (expected ~0.9710)")
print(f"Uniform 3-cls: H = {entropy(y_uniform):.4f}  (expected {np.log2(3):.4f} = log2(3))")

We compare the Gini and Entropy curves on the same axes (entropy scaled by 1/2 to align ranges). The two curves are nearly identical, which is why both criteria lead to nearly identical trees in practice.

In [ ]:
# Compare Gini and Entropy curves side-by-side (normalized)
p = np.linspace(1e-9, 1 - 1e-9, 500)

gini_vals    = 2 * p * (1 - p)
entropy_vals = -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

plt.figure(figsize=(8, 4))
plt.plot(p, gini_vals,          'b-',  linewidth=2, label='Gini = 2p(1-p)')
plt.plot(p, entropy_vals / 2,   'r--', linewidth=2, label='Entropy / 2  (rescaled to [0, 0.5])')
plt.xlabel('p (proportion of positive class)')
plt.ylabel('Impurity')
plt.title('Gini vs. Entropy — Binary Classification')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Note: the two curves are nearly identical. Gini and Entropy give very similar splits.")

<a id="variance"></a>
## 7. Splitting Criterion 3: Variance Reduction (Regression)

### Intuition

For regression tasks (predicting a continuous value), we cannot use Gini or Entropy. Instead, we measure how **spread out** the target values are within each node.

If a node contains samples with very similar $y$ values (low variance), predictions will be accurate — we can confidently predict their mean.

We want splits that create children with **low variance** — i.e., each child contains samples whose $y$ values are close together.

### Formula

Impurity = variance of target values:

$$\text{Var}(D) = \frac{1}{|D|} \sum_{i \in D} (y_i - \bar{y})^2$$

Variance Reduction (the gain we maximize):

$$\text{VR}(D, j, t) = \text{Var}(D) - \frac{|D_L|}{|D|} \text{Var}(D_L) - \frac{|D_R|}{|D|} \text{Var}(D_R)$$

Leaf prediction: the **mean** of $y$ values in the leaf.

We implement `variance_impurity` as a thin wrapper around `np.var`. This is all that is needed to swap the classification impurity functions for a regression impurity function.

In [ ]:
def variance_impurity(y):
    """
    Variance of target values y. Used as impurity for regression trees.
    Returns 0 if all values are equal.
    """
    if len(y) == 0:
        return 0.0
    return np.var(y)

We compare variance on a tightly-clustered array (values near 2.0) versus a spread-out array (values ranging 1–8), confirming that the impurity measure correctly ranks the two groups.

In [ ]:
y_tight  = np.array([2.1, 2.0, 1.9, 2.2, 2.0])
y_spread = np.array([1.0, 5.0, 2.0, 8.0, 3.0])

print(f"Tight values:  Var = {variance_impurity(y_tight):.4f}")
print(f"Spread values: Var = {variance_impurity(y_spread):.4f}")

<a id="split-gain"></a>
## 8. Computing the Gain of a Split

Given an impurity function $I$, the **gain** of splitting dataset $D$ at feature $j$ and threshold $t$ is:

$$\text{Gain}(D, j, t) = I(D) - \frac{|D_L|}{|D|} I(D_L) - \frac{|D_R|}{|D|} I(D_R)$$

The weighted average of children's impurity is called **weighted impurity** or **conditional impurity**. Gain = parent impurity − weighted child impurity.

We want to **maximize** this gain.

`split_gain` is a generic function that accepts any impurity callable. It computes the weighted average child impurity and subtracts it from the parent impurity.

In [ ]:
def split_gain(y, y_left, y_right, impurity_fn):
    """
    Information gain (impurity reduction) of a split.
    
    Args:
        y:            labels/values at current node
        y_left:       labels/values going to left child
        y_right:      labels/values going to right child
        impurity_fn:  callable — gini_impurity, entropy, or variance_impurity
    """
    n, n_l, n_r = len(y), len(y_left), len(y_right)
    if n == 0 or n_l == 0 or n_r == 0:
        return 0.0
    parent_impurity = impurity_fn(y)
    weighted_child  = (n_l / n) * impurity_fn(y_left) + (n_r / n) * impurity_fn(y_right)
    return parent_impurity - weighted_child

We test `split_gain` on three qualitatively different splits of the same parent node — a perfect split (gain = maximum), a useless split (gain = 0), and a partial split — using both Gini and Entropy.

In [ ]:
# Concrete example: 6 samples, binary labels
y_parent = [0, 0, 0, 1, 1, 1]   # 50/50 split → high impurity

# Split A: perfect — each side is pure
y_A_left, y_A_right = [0, 0, 0], [1, 1, 1]
# Split B: no improvement — each side still 50/50
y_B_left, y_B_right = [0, 1, 0], [1, 0, 1]
# Split C: partial improvement
y_C_left, y_C_right = [0, 0, 1], [0, 1, 1]

for name, yl, yr in [('A (perfect)', y_A_left, y_A_right),
                      ('B (useless)',  y_B_left, y_B_right),
                      ('C (partial)',  y_C_left, y_C_right)]:
    g_gini = split_gain(y_parent, yl, yr, gini_impurity)
    g_entr = split_gain(y_parent, yl, yr, entropy)
    print(f"Split {name}:  Gini gain = {g_gini:.4f}   Entropy gain = {g_entr:.4f}")

<a id="best-split"></a>
## 9. Finding the Best Split

We have $n$ samples and $p$ features. For each feature, we try every unique value as a threshold. This is an $O(np)$ search per node.

**Algorithm:**
```
best_gain = -∞
for each feature j in {0, ..., p-1}:
    sort unique values of X[:, j]
    for each threshold t (midpoints between consecutive sorted values):
        split D into D_L and D_R
        gain = Gain(D, j, t)
        if gain > best_gain:
            best_gain = gain
            best_feature = j
            best_threshold = t
return best_feature, best_threshold
```

Using midpoints between sorted values (rather than the values themselves) is standard — it ensures the threshold is not data-dependent in a fragile way.

`best_split` iterates over every feature and every candidate threshold (midpoints of consecutive unique values), evaluates the gain at each, and returns the best `(feature, threshold, gain)` triple.

In [ ]:
def best_split(X, y, impurity_fn):
    """
    Exhaustive search for the best (feature, threshold) split.
    
    Returns:
        best_feature_idx:  column index of best feature
        best_threshold:    threshold value
        best_gain:         impurity gain of the split
    """
    n_samples, n_features = X.shape
    best_gain      = -np.inf
    best_feature   = None
    best_threshold = None

    for feature_idx in range(n_features):
        col = X[:, feature_idx]
        sorted_vals = np.sort(np.unique(col))
        thresholds  = (sorted_vals[:-1] + sorted_vals[1:]) / 2.0

        for t in thresholds:
            left_mask  = col <= t
            right_mask = ~left_mask

            if left_mask.sum() == 0 or right_mask.sum() == 0:
                continue

            gain = split_gain(y, y[left_mask], y[right_mask], impurity_fn)

            if gain > best_gain:
                best_gain      = gain
                best_feature   = feature_idx
                best_threshold = t

    return best_feature, best_threshold, best_gain

We run `best_split` on a small 5-sample toy dataset with 2 features and 2 classes, then print the resulting partition to verify correctness.

In [ ]:
# Quick test
X_toy = np.array([[2.5, 1.0],
                  [1.0, 2.0],
                  [3.0, 1.5],
                  [1.5, 3.0],
                  [2.0, 2.5]])
y_toy = np.array([0, 0, 1, 1, 1])

feat, thresh, gain = best_split(X_toy, y_toy, gini_impurity)
print(f"Best split: feature {feat}, threshold {thresh:.2f}, gain {gain:.4f}")
print(f"  Left  ({np.sum(X_toy[:, feat] <= thresh)} samples): {y_toy[X_toy[:, feat] <= thresh]}")
print(f"  Right ({np.sum(X_toy[:, feat] > thresh)} samples):  {y_toy[X_toy[:, feat] > thresh]}")

<a id="recursive-build"></a>
## 10. Building the Tree Recursively

### The Recursive Algorithm (CART — Classification and Regression Trees)

```
function build_tree(D, depth):
    if stopping_criterion(D, depth):
        return Leaf(prediction(D))
    
    j*, t* = argmax_{j,t} Gain(D, j, t)
    D_L = { (x,y) in D : x[j*] <= t* }
    D_R = { (x,y) in D : x[j*]  > t* }
    
    return InternalNode(
        feature   = j*,
        threshold = t*,
        left      = build_tree(D_L, depth + 1),
        right     = build_tree(D_R, depth + 1)
    )
```

### Stopping Criteria

Without stopping criteria, the tree grows until every leaf is pure — this **overfits** the training data.

Common stopping rules (use any combination):

| Criterion | Description |
|---|---|
| `max_depth` | Stop if depth exceeds limit |
| `min_samples_split` | Stop if node has too few samples to split |
| `min_samples_leaf` | Stop if a resulting child would be too small |
| `min_impurity_decrease` | Stop if gain is below threshold |
| **Pure node** | Stop if node is already pure (gain = 0) |

### Leaf Predictions

- **Classification**: majority class in the leaf
- **Regression**: mean of $y$ values in the leaf

The `Node` class is a simple container. Internal nodes store the feature index and threshold used for splitting, plus references to left and right children. Leaf nodes store only a predicted value.

In [ ]:
class Node:
    """Represents a single node in the decision tree."""
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature   = feature    # feature index to split on (None if leaf)
        self.threshold = threshold  # split threshold (None if leaf)
        self.left      = left       # left child Node
        self.right     = right      # right child Node
        self.value     = value      # prediction value (set only for leaf nodes)

    @property
    def is_leaf(self):
        return self.value is not None


print("Node class defined.")

<a id="implementation"></a>
## 11. Complete Decision Tree Implementation from Scratch

We now assemble everything into a `DecisionTree` class that supports both **classification** and **regression**.

The `DecisionTree` class wraps `best_split` and `Node` into a sklearn-style `fit` / `predict` API. It dispatches to the correct impurity function based on the `criterion` parameter and applies the stopping criteria at each recursive call.

In [ ]:
class DecisionTree:
    """
    Decision tree for classification (Gini or Entropy) or regression (Variance).

    Parameters
    ----------
    task : 'classification' or 'regression'
    criterion : 'gini', 'entropy', or 'variance'
    max_depth : int or None
    min_samples_split : int  — minimum samples required to attempt a split
    min_impurity_decrease : float  — minimum gain required to perform a split
    """

    CRITERIA = {
        'gini':     gini_impurity,
        'entropy':  entropy,
        'variance': variance_impurity,
    }

    def __init__(self,
                 task='classification',
                 criterion='gini',
                 max_depth=None,
                 min_samples_split=2,
                 min_impurity_decrease=0.0):
        self.task                  = task
        self.criterion             = criterion
        self.impurity_fn           = self.CRITERIA[criterion]
        self.max_depth             = max_depth
        self.min_samples_split     = min_samples_split
        self.min_impurity_decrease = min_impurity_decrease
        self.root                  = None

    # ------------------------------------------------------------------
    # Training
    # ------------------------------------------------------------------

    def fit(self, X, y):
        """Build the decision tree from training data."""
        self.n_features_ = X.shape[1]
        self.root = self._build(X, y, depth=0)
        return self

    def _build(self, X, y, depth):
        """Recursively build the tree. Returns a Node."""

        # --- Stopping criteria ---
        if self._should_stop(X, y, depth):
            return Node(value=self._leaf_value(y))

        # --- Find best split ---
        feature, threshold, gain = best_split(X, y, self.impurity_fn)

        # No beneficial split found
        if feature is None or gain < self.min_impurity_decrease:
            return Node(value=self._leaf_value(y))

        # --- Partition data ---
        mask_left  = X[:, feature] <= threshold
        mask_right = ~mask_left

        left_child  = self._build(X[mask_left],  y[mask_left],  depth + 1)
        right_child = self._build(X[mask_right], y[mask_right], depth + 1)

        return Node(feature=feature, threshold=threshold,
                    left=left_child, right=right_child)

    def _should_stop(self, X, y, depth):
        """True if any stopping criterion is met."""
        if self.max_depth is not None and depth >= self.max_depth:
            return True
        if len(y) < self.min_samples_split:
            return True
        if len(np.unique(y)) == 1:
            return True
        return False

    def _leaf_value(self, y):
        """Prediction for a leaf: majority class or mean."""
        if self.task == 'classification':
            return Counter(y).most_common(1)[0][0]
        else:
            return np.mean(y)

    # ------------------------------------------------------------------
    # Inference
    # ------------------------------------------------------------------

    def predict(self, X):
        """Predict class labels (or values) for each row of X."""
        return np.array([self._traverse(x, self.root) for x in X])

    def _traverse(self, x, node):
        """Walk the tree for a single sample x, return leaf value."""
        if node.is_leaf:
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse(x, node.left)
        else:
            return self._traverse(x, node.right)

    # ------------------------------------------------------------------
    # Inspection helpers
    # ------------------------------------------------------------------

    def depth(self):
        """Maximum depth of the built tree."""
        return self._node_depth(self.root)

    def _node_depth(self, node):
        if node is None or node.is_leaf:
            return 0
        return 1 + max(self._node_depth(node.left), self._node_depth(node.right))

    def n_leaves(self):
        """Total number of leaf nodes."""
        return self._count_leaves(self.root)

    def _count_leaves(self, node):
        if node is None:
            return 0
        if node.is_leaf:
            return 1
        return self._count_leaves(node.left) + self._count_leaves(node.right)

    def print_tree(self, node=None, prefix='', indent='  '):
        """Pretty-print the tree structure."""
        if node is None:
            node = self.root
        if node.is_leaf:
            print(f"{prefix}Leaf → predict {node.value}")
        else:
            print(f"{prefix}[feature {node.feature} <= {node.threshold:.3f}]")
            print(f"{prefix}{indent}True:")
            self.print_tree(node.left,  prefix + indent * 2)
            print(f"{prefix}{indent}False:")
            self.print_tree(node.right, prefix + indent * 2)


print("DecisionTree class defined.")

<a id="prediction"></a>
## 12. Prediction: Traversing the Tree

Once trained, predicting for a new sample $x$ is simple: start at the root and follow branches based on $x$'s feature values until reaching a leaf.

```
function predict(x, node):
    if node is leaf:
        return node.value
    if x[node.feature] <= node.threshold:
        return predict(x, node.left)
    else:
        return predict(x, node.right)
```

**Time complexity:** $O(\text{depth})$ per prediction — extremely fast at inference time.

<a id="classification-demo"></a>
## 13. Demo: Classification (Iris Dataset)

The Iris dataset has 150 samples, 4 features, and 3 classes (flower species).

We load the Iris dataset and split it into an 80/20 train-test partition. This dataset is small enough that we can train our from-scratch implementation in seconds.

In [ ]:
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42
)

print(f"Training: {X_train.shape[0]} samples,  Test: {X_test.shape[0]} samples")
print(f"Features: {iris.feature_names}")
print(f"Classes:  {iris.target_names}")

We train our custom tree with both Gini and Entropy criteria (max depth 4) and compare accuracy, depth, and leaf count against sklearn's `DecisionTreeClassifier`.

In [ ]:
# Train our custom tree with different criteria
for criterion in ['gini', 'entropy']:
    tree = DecisionTree(task='classification', criterion=criterion, max_depth=4)
    tree.fit(X_train, y_train)
    acc = accuracy_score(y_test, tree.predict(X_test))
    print(f"Our tree  criterion={criterion:8s}  Accuracy={acc:.4f}  Depth={tree.depth()}  Leaves={tree.n_leaves()}")

print()
for criterion in ['gini', 'entropy']:
    sk = DecisionTreeClassifier(criterion=criterion, max_depth=4, random_state=42)
    sk.fit(X_train, y_train)
    acc = accuracy_score(y_test, sk.predict(X_test))
    print(f"sklearn   criterion={criterion:8s}  Accuracy={acc:.4f}  Depth={sk.get_depth()}  Leaves={sk.get_n_leaves()}")

We train a shallower tree on only 2 features (petal length and petal width) and print the full tree structure to make it easy to read the learned decision rules.

In [ ]:
# Print learned tree structure on 2 features for readability
tree_2f = DecisionTree(task='classification', criterion='gini', max_depth=3)
tree_2f.fit(X_train[:, 2:], y_train)   # petal length and petal width only
print("Tree structure (features: [petal_length, petal_width]):")
tree_2f.print_tree()

<a id="decision-boundary"></a>
## 14. Visualizing the Decision Boundary

Using only 2 features lets us plot the decision boundary — the regions in feature space assigned to each class.

`plot_decision_boundary` creates a dense meshgrid over the 2D feature space, predicts the class at every grid point, and uses `contourf` to shade the resulting regions. Training points are overlaid on top.

In [ ]:
def plot_decision_boundary(tree, X, y, feature_names=None, title='Decision Boundary'):
    """Plot 2D decision boundary for a trained decision tree."""
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                          np.arange(y_min, y_max, h))

    Z = tree.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    colors      = ['#AECDE8', '#FFCBA4', '#C3E8C3']
    point_colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.contourf(xx, yy, Z, alpha=0.4, levels=[-0.5, 0.5, 1.5, 2.5], colors=colors)
    for cls, col in zip(np.unique(y), point_colors):
        mask = y == cls
        ax.scatter(X[mask, 0], X[mask, 1], c=col, label=f'Class {cls}',
                   edgecolors='k', linewidth=0.5, s=50)

    if feature_names:
        ax.set_xlabel(feature_names[0])
        ax.set_ylabel(feature_names[1])
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()

We train three trees at increasing depth limits (2, 4, 20) on the petal features and plot each decision boundary in turn. This visually demonstrates underfitting, a good fit, and overfitting.

In [ ]:
X_2d = X_iris[:, 2:]   # petal length, petal width

for max_d, label in [(2,  'max_depth=2  (underfitting)'),
                      (4,  'max_depth=4  (good fit)'),
                      (20, 'max_depth=20 (overfitting)')]:
    t = DecisionTree(task='classification', criterion='gini', max_depth=max_d)
    t.fit(X_2d, y_iris)
    plot_decision_boundary(t, X_2d, y_iris,
                           feature_names=['Petal Length', 'Petal Width'],
                           title=label + f'  (depth={t.depth()}, leaves={t.n_leaves()})')

<a id="overfitting"></a>
## 15. Overfitting vs. Regularization

The visualizations above show a key challenge: **without depth limits, decision trees memorize the training data** (axis-aligned boundaries fragment finely around every training point).

### Effect of `max_depth` on accuracy

We sweep `max_depth` from 1 to 14, recording train and test accuracy at each depth. The resulting plot shows the classic bias-variance trade-off: train accuracy rises monotonically while test accuracy peaks and then plateaus or drops.

In [ ]:
depths = range(1, 15)
train_accs, test_accs = [], []

for d in depths:
    t = DecisionTree(task='classification', criterion='gini', max_depth=d)
    t.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, t.predict(X_train)))
    test_accs.append( accuracy_score(y_test,  t.predict(X_test)))

plt.figure(figsize=(8, 4))
plt.plot(depths, train_accs, 'b-o', label='Train accuracy')
plt.plot(depths, test_accs,  'r-o', label='Test accuracy')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Overfitting: Train vs Test Accuracy as Tree Depth Increases')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_depth = list(depths)[np.argmax(test_accs)]
print(f"Best test accuracy at depth {best_depth}: {max(test_accs):.4f}")

<a id="regression-demo"></a>
## 16. Demo: Regression

The same algorithm applies to regression: replace the impurity measure with variance and the leaf prediction with the mean.

We generate a 1D sine-plus-noise dataset and fit regression trees at three depth limits (2, 5, 15). Each subplot shows the piecewise-constant approximation alongside the true sine curve and its RMSE on the held-out test set.

In [ ]:
# 1D regression: sine + noise
X_reg = np.sort(np.random.uniform(0, 2 * np.pi, 120)).reshape(-1, 1)
y_reg = np.sin(X_reg.ravel()) + np.random.normal(0, 0.3, 120)

X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.25, random_state=0)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, max_d in zip(axes, [2, 5, 15]):
    tree_reg = DecisionTree(task='regression', criterion='variance', max_depth=max_d)
    tree_reg.fit(X_tr, y_tr)

    X_plot = np.linspace(0, 2 * np.pi, 400).reshape(-1, 1)
    y_plot = tree_reg.predict(X_plot)
    rmse   = np.sqrt(mean_squared_error(y_te, tree_reg.predict(X_te)))

    ax.scatter(X_tr, y_tr, s=15, alpha=0.6, label='Train data')
    ax.scatter(X_te, y_te, s=15, alpha=0.6, c='orange', label='Test data')
    ax.plot(X_plot, y_plot, 'r-', linewidth=2, label='Prediction')
    ax.plot(X_plot, np.sin(X_plot), 'g--', linewidth=1, alpha=0.7, label='True sin(x)')
    ax.set_title(f'max_depth={max_d}  (RMSE={rmse:.3f})')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Regression Tree: Underfitting → Good Fit → Overfitting', y=1.02)
plt.tight_layout()
plt.show()

<a id="feature-importance"></a>
## 17. Feature Importance

A decision tree gives us a natural measure of **feature importance**: how much does each feature contribute to reducing impurity across all splits?

$$\text{Importance}(j) = \sum_{\text{node } v \text{ splitting on } j} \frac{n_v}{n} \cdot \text{Gain}(D_v, j, t_v)$$

Features that appear in high-gain, high-sample-count nodes get high importance scores. These are then normalized to sum to 1.

`compute_feature_importances` performs a recursive tree traversal, accumulating the sample-weighted gain at every internal node and assigning it to the splitting feature. The final array is normalized to sum to 1.

In [ ]:
def compute_feature_importances(tree_obj, n_features, X_total, y_total):
    """
    Accumulate impurity-weighted gains across all internal nodes.
    Returns a normalized array of shape (n_features,).
    """
    importances = np.zeros(n_features)
    n_total = len(y_total)

    def recurse(node, X_node, y_node):
        if node.is_leaf or len(y_node) == 0:
            return
        mask_l = X_node[:, node.feature] <= node.threshold
        X_l, y_l = X_node[mask_l],  y_node[mask_l]
        X_r, y_r = X_node[~mask_l], y_node[~mask_l]

        gain = split_gain(y_node, y_l, y_r, tree_obj.impurity_fn)
        importances[node.feature] += (len(y_node) / n_total) * gain

        recurse(node.left,  X_l, y_l)
        recurse(node.right, X_r, y_r)

    recurse(tree_obj.root, X_total, y_total)

    total = importances.sum()
    if total > 0:
        importances /= total
    return importances

We fit a depth-5 tree on the full Iris dataset and compare the resulting feature importances against sklearn's implementation using horizontal bar charts.

In [ ]:
tree_full = DecisionTree(task='classification', criterion='gini', max_depth=5)
tree_full.fit(X_iris, y_iris)
importances = compute_feature_importances(tree_full, X_iris.shape[1], X_iris, y_iris)

sk_full = DecisionTreeClassifier(criterion='gini', max_depth=5, random_state=42)
sk_full.fit(X_iris, y_iris)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, imp, title in zip(axes,
                           [importances, sk_full.feature_importances_],
                           ['Our Implementation', 'sklearn']):
    ax.barh(iris.feature_names, imp, color='steelblue')
    ax.set_xlabel('Importance')
    ax.set_title(f'Feature Importances — {title}')
    ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

<a id="complexity"></a>
## 18. Complexity Analysis

| Phase | Time Complexity | Notes |
|---|---|---|
| **Training** | $O(n p d \log n)$ | $n$ = samples, $p$ = features, $d$ = depth; sorting once per feature per level |
| **Prediction** | $O(d)$ per sample | Walk tree from root to leaf |
| **Space** | $O(2^d)$ leaves | Exponential in depth, but depth is usually small |

In practice, sklearn uses optimized sorting and pre-sorting strategies, but the conceptual complexity is as above.

<a id="pros-cons"></a>
## 19. Advantages and Disadvantages

### Advantages

| Property | Detail |
|---|---|
| **Interpretability** | Easy to visualize and explain — humans can follow the decision path |
| **No feature scaling** | Splits are threshold-based; standardizing features makes no difference |
| **Handles mixed types** | Can split on categorical and continuous features |
| **Non-parametric** | No distributional assumptions |
| **Feature selection built-in** | Irrelevant features are simply not used |

### Disadvantages

| Property | Detail |
|---|---|
| **Overfitting** | Unlimited trees memorize training data; requires careful regularization |
| **High variance** | Small changes in data can produce very different trees (instability) |
| **Axis-aligned splits** | Cannot capture diagonal or curved decision boundaries efficiently |
| **Greedy** | No global optimum guarantee; local best split ≠ global best tree |

These disadvantages are addressed by **ensemble methods**:
- **Random Forest**: average many trees trained on random subsets of data and features → lower variance
- **Gradient Boosting (XGBoost, LightGBM)**: train trees sequentially to correct previous errors → lower bias

<a id="summary"></a>
## 20. Summary

What we built, step by step:

```
1. Impurity measures
   ├── Gini:     I(D) = 1 - Σ pₖ²
   ├── Entropy:  I(D) = -Σ pₖ log₂ pₖ
   └── Variance: I(D) = Var(y)

2. Split gain
   Gain(D, j, t) = I(D) - |DL|/|D| · I(DL) - |DR|/|D| · I(DR)

3. Best split search
   Brute-force over all (feature, threshold) pairs → O(np) per node

4. Recursive tree building (CART)
   Top-down, greedy, with stopping criteria (max_depth, min_samples_split)

5. Prediction
   Traverse tree from root to leaf — O(depth) per sample

6. Feature importance
   Weighted impurity reduction aggregated per feature, normalized to sum=1
```

Decision trees are the building block of most state-of-the-art tabular ML models (Random Forest, XGBoost, LightGBM). Understanding them deeply is the foundation for understanding ensembles.

<a id="sources"></a>
## 21. Sources

### Foundational Papers
- **Shannon, C.E. (1948)**. A Mathematical Theory of Communication. *Bell System Technical Journal*, 27, 379–423. — Foundation of information theory and entropy. [PDF](https://people.math.harvard.edu/~ctm/home/text/others/shannon/entropy/entropy.pdf)
- **Breiman, Friedman, Olshen, Stone (1984)**. *Classification and Regression Trees*. Chapman and Hall. — Original CART algorithm paper.
- **Quinlan, J.R. (1986)**. Induction of Decision Trees. *Machine Learning*, 1(1), 81–106. — Introduced ID3 algorithm. [DOI](https://doi.org/10.1007/BF00116251)
- **Quinlan, J.R. (1993)**. *C4.5: Programs for Machine Learning*. Morgan Kaufmann. — Extension of ID3 using entropy.

### Textbooks
- James, Witten, Hastie, Tibshirani. ***An Introduction to Statistical Learning***. Chapter 8: Tree-Based Methods. [Free PDF](https://www.statlearning.com/)
- Hastie, Tibshirani, Friedman. ***The Elements of Statistical Learning***. Chapter 9. [Free PDF](https://web.stanford.edu/~hastie/ElemStatLearn/)

### Online Resources
- [scikit-learn — Decision Trees](https://scikit-learn.org/stable/modules/tree.html)
- [Wikipedia — Decision tree learning](https://en.wikipedia.org/wiki/Decision_tree_learning)
- [Wikipedia — Information gain in decision trees](https://en.wikipedia.org/wiki/Information_gain_in_decision_trees)
- [Wikipedia — Gini impurity](https://en.wikipedia.org/wiki/Decision_tree_learning#Gini_impurity)